In [2]:
from pathlib import Path
import csv

# 使用你提供的正确路径
DATA_DIR = Path(r"C:\Users\Y\Desktop\data")

# 1. 读取训练集
train_lines = (DATA_DIR / "train.txt").read_text(encoding="utf-8").splitlines()
train_sents = [line.split() for line in train_lines if line.strip()]

# 2. 读取验证集、测试集、挑战集
def read_csv(filename):
    with open(DATA_DIR / filename, encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

dev_data = read_csv("dev.csv")
test_data = read_csv("test.csv")
challenge_data = read_csv("challenge.csv")

print("✅ 数据加载成功！")
print("训练集句子数:", len(train_sents))
print("验证题数:", len(dev_data))
print("测试题数:", len(test_data))
print("挑战题数:", len(challenge_data))

✅ 数据加载成功！
训练集句子数: 72
验证题数: 12
测试题数: 12
挑战题数: 6


In [3]:
from collections import Counter
import math

# 玩具语料
toy_text = """
学校 今天 发布 通知
学校 明天 发布 通知
学生 今天 提交 报告
学生 明天 提交 报告
教师 今天 发布 作业
教师 明天 提交 作业"""
toy_sents = [line.split() for line in toy_text.strip().split('\n')]

# 统计 N-gram
def build_ngrams(sentences, n):
    counts, hist_counts = Counter(), Counter()
    for words in sentences:
        seq = ["<s>"] * (n - 1) + words + ["</s>"]
        for i in range(len(seq) - n + 1):
            counts[tuple(seq[i:i+n])] += 1
            if n > 1: hist_counts[tuple(seq[i:i+n-1])] += 1
    return counts, hist_counts

uni_counts, _ = build_ngrams(toy_sents, 1)
bi_counts, bi_hist = build_ngrams(toy_sents, 2)
tri_counts, tri_hist = build_ngrams(toy_sents, 3)

# 平滑条件概率
def prob_smoothed(w, h, counts, hist_counts, k, vocab_size):
    c_hw = counts[h + (w,)]
    c_h = hist_counts.get(h, 0)
    return (c_hw + k) / (c_h + k * vocab_size)

print("=== 任务一：计数结果 ===")
print("发布 次数:", uni_counts[('发布',)])
print("发布 通知 次数:", bi_counts[('发布', '通知')])
print("今天 发布 通知 次数:", tri_counts[('今天', '发布', '通知')])

print("\n=== 任务一：条件概率结果 ===")
print("P(发布|今天) =", prob_smoothed('发布', ('今天',), bi_counts, bi_hist, k=0, vocab_size=12))
print("P(通知|发布) =", prob_smoothed('通知', ('发布',), bi_counts, bi_hist, k=0, vocab_size=12))
print("P(发布|学校 今天) =", prob_smoothed('发布', ('学校', '今天'), tri_counts, tri_hist, k=0, vocab_size=12))

=== 任务一：计数结果 ===
发布 次数: 3
发布 通知 次数: 2
今天 发布 通知 次数: 1

=== 任务一：条件概率结果 ===
P(发布|今天) = 0.6666666666666666
P(通知|发布) = 0.6666666666666666
P(发布|学校 今天) = 1.0


In [4]:
# 1. 建立项目词表
vocab = set()
for s in train_sents: vocab.update(s)
vocab.add("</s>")
vocab.add("<unk>")
V_size = len(vocab)

# 2. 建立 Bigram 模型
proj_bi_counts, proj_bi_hist = build_ngrams(train_sents, 2)

# 3. 句子评分函数（加一平滑）
def score_sentence(sent, counts, hist_counts, k=1.0):
    seq = ["<s>"] + sent + ["</s>"]
    score = 0
    for i in range(1, len(seq)):
        w, h = seq[i], (seq[i-1],)
        p = prob_smoothed(w, h, counts, hist_counts, k, V_size)
        score += math.log(p)
    return score

# 4. 单题预测与排序
def predict_row(row):
    sentence = row['sentence'].split()
    blank_idx = sentence.index('____')
    options = [row[f'option_{x}'] for x in 'ABCD']
    scores = {}
    
    for opt in options:
        words = sentence.copy()
        # OOV 映射
        words[blank_idx] = opt if opt in vocab else '<unk>' 
        scores[opt] = score_sentence(words, proj_bi_counts, proj_bi_hist, k=1.0)
        
    sorted_opts = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    best_score = sorted_opts[0][1]
    # 判断并列 (容差 1e-10)
    ties = [opt for opt, s in scores.items() if abs(s - best_score) < 1e-10]
    return sorted_opts, ties

print("=== 验证集预测结果 ===")
correct = 0
for row in dev_data:
    sorted_opts, ties = predict_row(row)
    pred_word = sorted_opts[0][0]
    if pred_word == row[f"option_{row['answer']}"]:
        correct += 1
print(f"验证集正确率: {correct} / {len(dev_data)} ({(correct/len(dev_data))*100:.2f}%)")

print("\n=== 测试集预测结果 ===")
correct_test = 0
for row in test_data:
    sorted_opts, ties = predict_row(row)
    pred_word = sorted_opts[0][0]
    if pred_word == row[f"option_{row['answer']}"]:
        correct_test += 1
print(f"测试集正确率: {correct_test} / {len(test_data)} ({(correct_test/len(test_data))*100:.2f}%)")

print("\n=== 挑战题 C04 输出 ===")
c04 = challenge_data[3] # 取 C04
opts, ties = predict_row(c04)
print(f"C04 候选词得分及排序: {opts}")
print(f"发生并列的候选词: {ties}")

=== 验证集预测结果 ===
验证集正确率: 12 / 12 (100.00%)

=== 测试集预测结果 ===
测试集正确率: 12 / 12 (100.00%)

=== 挑战题 C04 输出 ===
C04 候选词得分及排序: [('评估', -26.710197106762237), ('核验', -26.710197106762237), ('汇总', -26.710197106762237), ('审查', -26.710197106762237)]
发生并列的候选词: ['评估', '核验', '汇总', '审查']


In [5]:
# 测试句子
test_sentence = "学校 今天 通知 报告".split()

# 1. 未平滑评分
def score_unsmoothed(sent, counts, hist_counts):
    seq = ["<s>"] + sent + ["</s>"]
    score = 0
    for i in range(1, len(seq)):
        w, h = seq[i], (seq[i-1],)
        if h not in hist_counts or counts[h + (w,)] == 0:
            return float('-inf')
        p = counts[h + (w,)] / hist_counts[h]
        score += math.log(p)
    return score

# 2. 加一平滑评分
def score_smoothed(sent, counts, hist_counts, k, vocab_size, n):
    seq = ["<s>"] * (n - 1) + sent + ["</s>"]
    score = 0
    for i in range(n - 1, len(seq)):
        w = seq[i]
        h = tuple(seq[i-n+1:i])
        c_hw = counts[h + (w,)]
        c_h = hist_counts.get(h, 0)
        p = (c_hw + k) / (c_h + k * vocab_size)
        score += math.log(p)
    return score

print("=== 平滑对比 ===")
print(f"测试句子: {' '.join(test_sentence)}")
print(f"[未平滑] Bigram 得分: {score_unsmoothed(test_sentence, bi_counts, bi_hist)}")
print(f"[未平滑] Trigram 得分: {score_unsmoothed(test_sentence, tri_counts, tri_hist)}")

print(f"[加一平滑] Bigram 得分: {score_smoothed(test_sentence, bi_counts, bi_hist, k=1.0, vocab_size=12, n=2):.4f}")
print(f"[加一平滑] Trigram 得分: {score_smoothed(test_sentence, tri_counts, tri_hist, k=1.0, vocab_size=12, n=3):.4f}")

=== 平滑对比 ===
测试句子: 学校 今天 通知 报告
[未平滑] Bigram 得分: -inf
[未平滑] Trigram 得分: -inf
[加一平滑] Bigram 得分: -10.6252
[加一平滑] Trigram 得分: -11.2724
